# 12. Multi-Agent Orchestration (코드 기반 멀티에이전트 오케스트레이션)

**04_Handoffs**에서는 LLM이 자율적으로 핸드오프를 결정하는 **LLM 기반 오케스트레이션**을 배웠습니다.
이번에는 **코드로 직접** 에이전트 흐름을 제어하는 패턴을 학습합니다.

### LLM 기반 vs 코드 기반 오케스트레이션

| 구분 | LLM 기반 (Handoffs) | 코드 기반 |
|------|-------------------|----------|
| 제어 주체 | LLM이 자율 판단 | 개발자가 코드로 제어 |
| 예측 가능성 | 낮음 (LLM 판단에 의존) | 높음 (결정적 흐름) |
| 유연성 | 높음 | 보통 |
| 비용/속도 | 변동적 | 예측 가능 |
| 적합한 경우 | 복잡한 대화, 분류가 어려운 경우 | 파이프라인, 배치 처리, 정해진 워크플로우 |

### 주요 패턴
1. **순차 체이닝 (Sequential Chaining)**: A → B → C
2. **병렬 실행 (Parallelization)**: A, B, C 동시 실행
3. **분류 → 라우팅 (Classification + Routing)**: 분류 후 전문 에이전트에 전달

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
import openai
import asyncio

Model = os.environ["OPENAI_MODEL"]

## 1. 순차 체이닝 (Sequential Chaining)

에이전트A의 출력을 에이전트B의 입력으로 전달하는 파이프라인입니다.
각 단계가 순서대로 실행됩니다.

```
[작성 에이전트] → 초안 → [편집 에이전트] → 수정본 → [번역 에이전트] → 최종 결과
```

In [3]:
from agents import Agent, Runner

# Step 1: 글 작성 에이전트
writer_agent = Agent(
    name="작성기",
    instructions="""당신은 기술 블로그 작성자입니다.
주어진 주제에 대해 3~4문장의 짧은 기술 소개글을 작성하세요.
전문 용어를 적절히 사용하되 이해하기 쉽게 작성하세요.""",
    model=Model,
)

# Step 2: 편집 에이전트
editor_agent = Agent(
    name="편집기",
    instructions="""당신은 전문 편집자입니다.
주어진 글을 다음 기준으로 수정하세요:
- 문법 오류 수정
- 문장 가독성 향상
- 불필요한 표현 제거
수정된 글만 출력하세요.""",
    model=Model,
)

# Step 3: 번역 에이전트
translator_agent = Agent(
    name="번역기",
    instructions="""당신은 한영 번역가입니다.
주어진 한국어 글을 자연스러운 영어로 번역하세요.
번역된 영어만 출력하세요.""",
    model=Model,
)

# 순차 체이닝 실행
# Step 1: 글 작성
write_result = await Runner.run(writer_agent, "에이전트 AI에 대해 소개글을 작성해주세요.")
print(f"[Step 1] 초안:\n{write_result.final_output}\n")

# Step 2: 편집 (이전 단계의 출력을 입력으로 전달)
edit_result = await Runner.run(editor_agent, write_result.final_output)
print(f"[Step 2] 편집본:\n{edit_result.final_output}\n")

# Step 3: 번역 (이전 단계의 출력을 입력으로 전달)
translate_result = await Runner.run(translator_agent, edit_result.final_output)
print(f"[Step 3] 번역본:\n{translate_result.final_output}")

[Step 1] 초안:
에이전트 AI(Agentic AI)는 목표를 전달받으면 작업을 단계별로 계획하고, 필요한 도구를 활용해 실행하는 인공지능입니다. 일반적인 생성형 AI가 주로 질문에 답하거나 콘텐츠를 만드는 데 그친다면, 에이전트 AI는 실행 결과를 확인하고 다음 행동을 조정할 수 있습니다. 예를 들어 여행 일정을 짤 때 항공편과 숙소를 검색하고 조건에 맞춰 비교하는 작업을 이어서 수행할 수 있습니다. 다만 자율성이 높은 만큼, 중요한 결정에는 사람의 검토와 적절한 권한 제한이 필요합니다.

[Step 2] 편집본:
에이전트 AI(Agentic AI)는 목표를 전달받으면 작업을 단계별로 계획하고 필요한 도구를 활용해 실행하는 인공지능입니다. 일반적인 생성형 AI가 주로 질문에 답하거나 콘텐츠를 만드는 데 그치는 반면, 에이전트 AI는 실행 결과를 확인해 다음 행동을 조정할 수 있습니다. 예를 들어 여행 일정을 짤 때 항공편과 숙소를 검색하고, 조건에 맞는 항목을 비교할 수 있습니다. 다만 자율성이 높은 만큼 중요한 결정에는 사람의 검토와 적절한 권한 제한이 필요합니다.

[Step 3] 번역본:
Agentic AI is artificial intelligence that takes a goal, plans the steps needed to achieve it, and uses the necessary tools to carry them out. While conventional generative AI mainly answers questions or creates content, agentic AI can check the results of its actions and adjust what it does next. For example, when planning a trip, it can search for flights and accommodations and compare options that meet your criteria. However,

## 2. 병렬 실행 (Parallelization)

서로 독립적인 에이전트를 `asyncio.gather()`로 **동시에** 실행하여 시간을 절약합니다.

```
        ┌→ [감성 분석 에이전트] → 결과1   ─┐
[입력] ──┼→ [키워드 추출 에이전트] → 결과2 ─┼→ [종합]
        └→ [요약 에이전트] → 결과3   ─────┘
```

In [4]:
# 3개의 독립적인 분석 에이전트
sentiment_agent = Agent(
    name="감성 분석기",
    instructions="""주어진 텍스트의 감성을 분석하세요.
결과 형식: "감성: [긍정/부정/중립], 신뢰도: [높음/보통/낮음], 이유: [간단한 설명]" """,
    model=Model,
)

keyword_agent = Agent(
    name="키워드 추출기",
    instructions="""주어진 텍스트에서 핵심 키워드 3~5개를 추출하세요.
결과 형식: "키워드: [키워드1], [키워드2], [키워드3], ..." """,
    model=Model,
)

summary_agent = Agent(
    name="요약기",
    instructions="""주어진 텍스트를 한 문장으로 요약하세요.
결과 형식: "요약: [한 문장 요약]" """,
    model=Model,
)

text_to_analyze = """
OpenAI의 Agent SDK는 개발자들이 AI 에이전트를 쉽게 만들 수 있도록 설계되었습니다.
핸드오프, 도구 사용, 가드레일 등의 기능을 제공하여 복잡한 AI 워크플로우를
간단한 코드로 구현할 수 있게 해줍니다. 특히 Python 개발자에게 친숙한 인터페이스를
제공하며, 프로덕션 환경에서의 안정성도 고려되어 있습니다.
"""

# asyncio.gather로 3개 에이전트를 동시에 실행
sentiment_result, keyword_result, summary_result = await asyncio.gather(
    Runner.run(sentiment_agent, text_to_analyze),
    Runner.run(keyword_agent, text_to_analyze),
    Runner.run(summary_agent, text_to_analyze),
)

print(f"{sentiment_result.final_output}")
print(f"{keyword_result.final_output}")
print(f"{summary_result.final_output}")

감성: 긍정, 신뢰도: 높음, 이유: 개발 편의성과 다양한 기능, 프로덕션 안정성을 긍정적으로 설명하고 있습니다.
키워드: OpenAI Agent SDK, AI 에이전트, 핸드오프, 도구 사용, 가드레일
요약: OpenAI의 Agent SDK는 핸드오프, 도구 사용, 가드레일 등을 통해 개발자가 Python으로 안정적인 AI 에이전트와 복잡한 워크플로우를 쉽게 구현하도록 돕습니다.


## 3. 분류 → 라우팅 (Classification + Routing)

**Structured Output**으로 입력을 분류한 후, 결과에 따라 적절한 전문 에이전트를 실행합니다.

```
[입력] → [분류 에이전트] → 카테고리 → [전문 에이전트 선택] → [전문 에이전트 실행] → [응답]
```

In [5]:
from pydantic import BaseModel

# 분류 결과를 위한 Pydantic 모델
class RequestClassification(BaseModel):
    category: str  # "기술지원", "결제문의", "일반문의"
    confidence: float  # 0.0 ~ 1.0
    reason: str

# 분류 에이전트
classifier_agent = Agent(
    name="요청 분류기",
    instructions="""고객의 요청을 다음 카테고리 중 하나로 분류하세요:
- "기술지원": 소프트웨어 오류, 설치 문제, 기술적 질문
- "결제문의": 요금, 환불, 결제 수단, 구독 관련
- "일반문의": 영업시간, 위치, 서비스 소개 등 일반 질문

category에 정확한 카테고리명을 넣고, confidence에 확신도(0~1)를 넣으세요.""",
    model=Model,
    output_type=RequestClassification,
)

# 전문 에이전트들
tech_support_agent = Agent(
    name="기술지원 전문가",
    instructions="당신은 기술지원 전문가입니다. 소프트웨어 문제를 해결하고 기술적 도움을 제공합니다.",
    model=Model,
)

billing_agent = Agent(
    name="결제 전문가",
    instructions="당신은 결제 전문가입니다. 요금, 환불, 구독 관련 질문에 답변합니다.",
    model=Model,
)

general_agent = Agent(
    name="일반 상담사",
    instructions="당신은 일반 상담사입니다. 서비스에 대한 일반적인 질문에 답변합니다.",
    model=Model,
)

# 카테고리 → 에이전트 매핑
agent_map = {
    "기술지원": tech_support_agent,
    "결제문의": billing_agent,
    "일반문의": general_agent,
}

In [6]:
# 분류 → 라우팅 실행
async def handle_customer_request(request: str):
    # Step 1: 분류
    classification_result = await Runner.run(classifier_agent, request)
    classification = classification_result.final_output
    print(f"분류 결과: {classification.category} (확신도: {classification.confidence:.0%})")
    print(f"   이유: {classification.reason}")

    # Step 2: 라우팅 - 분류 결과에 따라 적절한 에이전트 선택
    selected_agent = agent_map.get(classification.category, general_agent)
    print(f"선택된 에이전트: {selected_agent.name}\n")

    # Step 3: 전문 에이전트 실행
    response = await Runner.run(selected_agent, request)
    print(f"응답: {response.final_output}")

    return response.final_output

In [7]:
# 테스트 1: 기술지원 질문
await handle_customer_request("프로그램이 갑자기 멈추고 에러 메시지가 떠요. 어떻게 해야 하나요?")

분류 결과: 기술지원 (확신도: 99%)
   이유: 프로그램이 멈추고 오류 메시지가 표시되는 소프트웨어 문제에 대한 문의입니다.
선택된 에이전트: 기술지원 전문가

응답: 갑자기 멈추면 먼저 **저장되지 않은 작업이 있는지** 확인하세요. 가능하다면 기다리거나 자동 저장 여부를 확인한 뒤, 종료해 주세요.

그다음 아래 순서로 점검해 보세요.

1. **에러 메시지를 캡처하거나 문구를 기록**합니다.
2. 프로그램을 완전히 종료한 뒤 다시 실행합니다. 반응이 없다면 강제 종료가 필요할 수 있지만, 저장되지 않은 내용은 사라질 수 있습니다.
3. 컴퓨터를 재시작하고, 프로그램과 운영체제에 **업데이트가 있는지** 확인합니다.
4. 저장 공간과 메모리가 부족하지 않은지 확인하고, 같은 작업을 반복했을 때 다시 멈추는지 살펴봅니다.
5. 계속 문제가 생기면 프로그램의 이름과 버전, 사용 중인 기기·운영체제, 에러 메시지 전문, 문제가 발생한 직전 상황을 알려 주세요. 그 정보를 바탕으로 더 구체적으로 안내해 드릴게요.


'갑자기 멈추면 먼저 **저장되지 않은 작업이 있는지** 확인하세요. 가능하다면 기다리거나 자동 저장 여부를 확인한 뒤, 종료해 주세요.\n\n그다음 아래 순서로 점검해 보세요.\n\n1. **에러 메시지를 캡처하거나 문구를 기록**합니다.\n2. 프로그램을 완전히 종료한 뒤 다시 실행합니다. 반응이 없다면 강제 종료가 필요할 수 있지만, 저장되지 않은 내용은 사라질 수 있습니다.\n3. 컴퓨터를 재시작하고, 프로그램과 운영체제에 **업데이트가 있는지** 확인합니다.\n4. 저장 공간과 메모리가 부족하지 않은지 확인하고, 같은 작업을 반복했을 때 다시 멈추는지 살펴봅니다.\n5. 계속 문제가 생기면 프로그램의 이름과 버전, 사용 중인 기기·운영체제, 에러 메시지 전문, 문제가 발생한 직전 상황을 알려 주세요. 그 정보를 바탕으로 더 구체적으로 안내해 드릴게요.'

In [8]:
# 테스트 2: 결제 질문
await handle_customer_request("지난달 결제가 두 번 된 것 같아요. 환불 가능한가요?")

분류 결과: 결제문의 (확신도: 99%)
   이유: 지난달 중복 결제 여부와 환불 가능성을 문의하고 있습니다.
선택된 에이전트: 결제 전문가

응답: 실제로 같은 결제가 두 번 청구된 것이 맞다면 환불 대상일 수 있습니다. 다만 한 건이 **승인 대기 중인 임시 승인**이라면 며칠 안에 자동으로 취소될 수도 있어요.

먼저 카드·은행 명세서에서 두 건의 **금액, 가맹점명, 결제일, 처리 상태**가 같은지 확인해 주세요. 중복 청구가 확정되면 해당 서비스의 고객지원에 두 거래 내역을 알려 환불을 요청하세요. 해결되지 않으면 카드사나 은행에도 이의를 제기할 수 있습니다.

어떤 서비스에서 결제됐는지와 두 거래의 날짜·금액을 알려주시면 다음 단계를 안내해 드릴게요. 카드번호나 보안코드 같은 민감한 정보는 보내지 마세요.


'실제로 같은 결제가 두 번 청구된 것이 맞다면 환불 대상일 수 있습니다. 다만 한 건이 **승인 대기 중인 임시 승인**이라면 며칠 안에 자동으로 취소될 수도 있어요.\n\n먼저 카드·은행 명세서에서 두 건의 **금액, 가맹점명, 결제일, 처리 상태**가 같은지 확인해 주세요. 중복 청구가 확정되면 해당 서비스의 고객지원에 두 거래 내역을 알려 환불을 요청하세요. 해결되지 않으면 카드사나 은행에도 이의를 제기할 수 있습니다.\n\n어떤 서비스에서 결제됐는지와 두 거래의 날짜·금액을 알려주시면 다음 단계를 안내해 드릴게요. 카드번호나 보안코드 같은 민감한 정보는 보내지 마세요.'

In [9]:
# 테스트 3: 일반 질문
await handle_customer_request("주말에도 고객센터가 운영되나요?")

분류 결과: 일반문의 (확신도: 99%)
   이유: 고객센터의 주말 운영 여부를 묻는 영업시간 관련 일반 문의입니다.
선택된 에이전트: 일반 상담사

응답: 서비스마다 운영 시간이 다릅니다. 어떤 서비스의 고객센터를 말씀하시는지 알려주시면 주말 운영 여부를 확인해 드릴게요.


'서비스마다 운영 시간이 다릅니다. 어떤 서비스의 고객센터를 말씀하시는지 알려주시면 주말 운영 여부를 확인해 드릴게요.'

## 4. 종합 패턴: 병렬 + 순차 조합

실전에서는 여러 패턴을 **조합**하여 사용합니다.

```
[고객 리뷰] ──→ [병렬 분석] ──→ [종합 에이전트] ──→ [최종 보고서]
                 ├ 감성 분석
                 ├ 키워드 추출
                 └ 요약
```

In [10]:
# 종합 보고서를 작성하는 에이전트
report_agent = Agent(
    name="보고서 작성기",
    instructions="""당신은 분석 보고서 작성자입니다.
주어진 분석 결과들을 종합하여 간결한 보고서를 작성하세요.
보고서 형식:
---
[분석 보고서]
- 요약: ...
- 감성: ...
- 핵심 키워드: ...
- 종합 의견: (위 분석을 바탕으로 한 줄 의견)
---""",
    model=Model,
)

customer_review = """
이 제품 정말 좋아요! 배송도 빠르고 포장도 꼼꼼했습니다.
다만 설명서가 영어로만 되어 있어서 처음에 좀 헷갈렸어요.
전체적으로는 가격 대비 성능이 훌륭하고, 다음에도 이 브랜드를 구매할 의향이 있습니다.
"""

# Step 1: 병렬 분석 (동시 실행)
print("Step 1: 병렬 분석 실행 중...\n")
s_result, k_result, sm_result = await asyncio.gather(
    Runner.run(sentiment_agent, customer_review),
    Runner.run(keyword_agent, customer_review),
    Runner.run(summary_agent, customer_review),
)

print(f"  감성: {s_result.final_output}")
print(f"  키워드: {k_result.final_output}")
print(f"  요약: {sm_result.final_output}")

# Step 2: 순차 체이닝 - 분석 결과를 종합
print(f"\nStep 2: 종합 보고서 작성 중...\n")
combined_input = f"""다음 분석 결과를 종합해주세요:

원본 리뷰: {customer_review}

분석 결과:
1. {s_result.final_output}
2. {k_result.final_output}
3. {sm_result.final_output}
"""

report_result = await Runner.run(report_agent, combined_input)
print(f"📊 최종 보고서:\n{report_result.final_output}")

Step 1: 병렬 분석 실행 중...

  감성: 감성: 긍정, 신뢰도: 높음, 이유: 설명서가 영어뿐이라 아쉬웠지만 제품 성능과 배송, 포장에 만족했고 재구매 의향도 있습니다.
  키워드: 키워드: 제품 만족도, 빠른 배송, 꼼꼼한 포장, 가격 대비 성능, 재구매 의향
  요약: 요약: 배송과 포장이 만족스럽고 가격 대비 성능이 뛰어나 재구매 의향이 있지만, 설명서가 영어로만 되어 있어 다소 헷갈렸습니다.

Step 2: 종합 보고서 작성 중...

📊 최종 보고서:
[분석 보고서]
- 요약: 배송과 포장이 만족스럽고 가격 대비 성능이 뛰어나 재구매 의향이 높지만, 영어 설명서가 다소 불편했습니다.
- 감성: 긍정적(신뢰도 높음)
- 핵심 키워드: 제품 만족도, 빠른 배송, 꼼꼼한 포장, 가격 대비 성능, 재구매 의향, 영어 설명서
- 종합 의견: 설명서의 언어 개선 여지는 있지만, 전반적으로 만족도가 높아 재구매 가능성이 큰 제품입니다.
---
